# Data Atelier - Prendre en main les méthodes d'explicabilités pour interpréter son modèle de machine learning !

Pour comprendre le fonctionnement des modèles de machine learning, il est proposé de comprendre un modèle appliqué au dataset en open data : [Baromètre Représentations sociales du changement climatique](https://data.ademe.fr/datasets/barometre-representations-sociales-du-changement-climatique)




### Pour une installation des dépendances à partir d'un environnement et d'un clone du repo
pip install -r ./chemin/requirements.dev.txt

## Import des librairies

In [ ]:
#Pour la manipulation de données
import catboost
import numpy as np
import pandas as pd

#Pour la modélisation
from category_encoders import OrdinalEncoder

#Pour l'explicabilité
from shapash.explainer.smart_explainer import SmartExplainer
from sklearn.model_selection import train_test_split


In [ ]:
#Pour afficher l'ensemble des colonnes et contenus des lignes
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

## Import du dataset et cleaning

In [ ]:
df = pd.read_csv("./data/BDD OpinionWay pour l'ADEME - Barometre changement climatique - Compilation 2023.csv", 
                       delimiter=';',
                       encoding='UTF-8-SIG')

In [ ]:
df.head(1)

### Création de variables

In [ ]:
# création d'une variable sur une note moyenne des actions individuelles que le sondé est prêt à faire
notes_mapping = {
    '...vous le faites déjà depuis un certain temps': 4,
    '...vous le faites déjà': 4,
    '...vous pourriez le faire assez facilement': 3,
    '...vous pourriez le faire mais difficilement': 2,
    '...vous ne pouvez pas le faire': 1
}

def calculer_moyenne(row):
    notes = [notes_mapping[val] for val in row if val != 'No rep.']
    return sum(notes) / len(notes) if notes else 0

df_q20 = df.filter(like='q20', axis=1).replace({'nan': 'No rep.', np.nan: 'No rep.'})

colonnes_action_individuelle = df_q20.filter(like='q20').drop(columns=['splitq20'])

df_q20['moyenne_action_individuelle'] = colonnes_action_individuelle.apply(calculer_moyenne, axis=1)

df_q20['moyenne_action_individuelle'].round().value_counts()

In [ ]:
# création d'une variable sur une note moyenne des mesures que le sondé considère souhaitable
notes_mapping = {
    'Très souhaitable': 4,
    'Assez souhaitable': 3,
    'Pas vraiment souhaitable': 2,
    'Pas du tout souhaitable': 1
}
def calculer_moyenne(row):
    notes = [notes_mapping[val] for val in row if val != 'No rep.']
    return sum(notes) / len(notes) if notes else 0

df_q21 = df.filter(like='q21', axis=1).replace({'nan': 'No rep.', np.nan: 'No rep.'})

colonnes_mesures_souhaitables = df_q21.filter(like='q21')

df_q21['moyenne_mesures_souhaitables'] = colonnes_mesures_souhaitables.apply(calculer_moyenne, axis=1)

df_q21['moyenne_mesures_souhaitables'].round().value_counts()

In [ ]:
#ajout des 2 variables construites au dataframe initial
df = pd.concat([df, df_q20[['moyenne_action_individuelle']], df_q21['moyenne_mesures_souhaitables']], axis=1)

### Simplification du dataset

In [ ]:
# Dictionnaire de renommage des colonnes pour avoir un nom plus simple et court
columns_to_rename = {'S1. genre' : 'genre',
                      'S2. âge' : 'tranche_age',
                      'S3. Zone géo' : 'zone_geo',
                      'Région' : 'region',
                      'Département' : 'departement',
                      'S7b. Taille commune' : 'taille_commune',
                      'S6. Situation professionnelle' : 'situation_pro',
                      'S5. CSP' : 'csp',
                      'q5. Certitude/hypothèse impact effet de serre' : 'impact_effet_de_serre',
                      'q9. Causes désordres du climat' : 'cause_desordre',
                      's11. Désordre climatique subi' : "changement_subi",
                      'q10. Causes anthropiques/naturelles du changement climatique' : 'cause_changement_climatique',
                      'q11. Solutions changement climatique' : 'solutions',
                      'q18_i1. Causes GES activités industrielles' : 'causes_ges_industries', 
                      'q18_i2. Causes GES transports' : 'causes_ges_transports',
                      'q18_i3. Causes GES bâtiments' : 'causes_ges_batiments',
                      'q18_i4. Causes GES agriculture' : 'causes_ges_agriculture',
                      "q18_i5. Causes GES centrales de production d'électricité au gaz, charbon ou fuel"  : 'causes_ges_energie',
                      'q18_i6. Causes GES traitement des déchets' : 'causes_ges_dechets',
                      'q18_i7. Causes GES destruction des forêts' : 'causes_ges_forets',
                      'q18_i8. Causes GES centrales nucléaires': 'causes_ges_nucleaires',
                      'q18_i9. Causes GES activité volcanique' : 'causes_ges_volcans',
                      'q18_i10. Causes GES bombes aérosols' : 'causes_ges_aerosols',
                      'q18_i11. Causes GES numérique' : 'causes_ges_numeriques',
                      's13. Nb membres foyer' :'nb_membres_foyer',
                      's15. Diplôme' :'diplome', 
                      's17. Type habitat' : 'type_habitat',
                      's18. Statut locataire/propriétaire' : 'locataire_proprietaire',
                      's22. Mode de transport' : 'mode_transport',
                      's23. Partis politiques' : 'partis_politiques', 
                      's24. Echiquier politique' : 'echiquier_politiques',
                      's25. Sympathie mouvements écologistes' : 'sympathie_ecologie',
                      'q23. Véracité scientifique' : 'veracite_scientifique',
                      'q14. Médiatisation changement climatique' : 'mediatisation',
                      'q8. Conséquences changement climatique' : 'futures_consequences',
                      'q16. Optimisme limitation changement climatique' : 'limitation_changement',
                      'q13total. Condition changements modérés' : 'changements_moderes',
                      "q15btotal. associations" : "associations_acteurs",
                      "q15total. collectivités locales" : "collectivites_acteurs",
                      'q21_i1. Mesures Baisse vitesse autoroute' : 'baisse_vitesse_autoroute',
                      'q21_i10. Mesures Favoriser véhicules peu polluants' : 'favoriser_mobilite_douce',
                      'q21_i11. Mesures Menu restauration collective' : 'mesure_restauration'
                      }

In [ ]:
#renommage des colonnes du dict
df.rename(columns=columns_to_rename, inplace=True)

In [ ]:
# Sélection de features à conserver pour le modèle
features = ['Vague',
            'genre',
            'tranche_age',      
            'region',     
            'taille_commune',
            'situation_pro',        
            'impact_effet_de_serre',
            'cause_desordre',
            'cause_changement_climatique',
            'solutions',        
            'diplome', 
            'sympathie_ecologie',
            'changement_subi',
            'veracite_scientifique',
            'mediatisation',
            'futures_consequences',
            'limitation_changement',       
           # 'moyenne_action_individuelle',
            'moyenne_mesures_souhaitables',
                      ]          

In [ ]:
#suppressions des lignes avec trop de valeurs vides et la cible "solutions" non répondu
print(df.shape)
df_reduit = df[features].copy()
df_reduit['number_of_NaNs'] = df_reduit.isnull().sum(axis=1)
df_reduit = df_reduit[df_reduit["number_of_NaNs"]<2]                      
df_reduit = df_reduit[df_reduit["solutions"] !="No rep."]
print(df_reduit.shape)

In [ ]:
#nous remplaçons les Nan pour les colonnes sélectionnées
df_reduit = df_reduit.fillna("No rep.")

### Renommage des modalités 
Nous renommons les modalités de certaines colonnes pour un meilleur affichage

In [ ]:
df_reduit.loc[df_reduit['cause_changement_climatique'] == 'Le réchauffement de la planète est causé par les activités humaines', 'cause_changement_climatique'] = 'activites_humaines'
df_reduit.loc[df_reduit['cause_changement_climatique'] == "Il s'agit uniquement d'un phénomène naturel qui a toujours existé", 'cause_changement_climatique'] = 'phenomene_naturel'

df_reduit.loc[df_reduit['impact_effet_de_serre'] == '...ou bien une certitude pour la plupart des scientifiques ?', 'impact_effet_de_serre'] = 'certitude_scientifique'
df_reduit.loc[df_reduit['impact_effet_de_serre'] == "...une hypothèse sur laquelle les scientifiques ne sont pas tous d'accord ?", 'impact_effet_de_serre'] = 'hypothese_non_certaine'

df_reduit.loc[df_reduit['sympathie_ecologie'] == '...assez de sympathie ?', 'sympathie_ecologie'] = 'assez'
df_reduit.loc[df_reduit['sympathie_ecologie'] == "...peu de sympathie ?", 'sympathie_ecologie'] = 'peu'
df_reduit.loc[df_reduit['sympathie_ecologie'] == "...beaucoup de sympathie ?", 'sympathie_ecologie'] = 'beaucoup'
df_reduit.loc[df_reduit['sympathie_ecologie'] == "...pas de sympathie ?", 'sympathie_ecologie'] = 'pas'

df_reduit.loc[df_reduit['cause_desordre'] == 'Les désordres du climat et leurs conséquences (tels que les canicules, les tempêtes, les sécheresses et les inondati', 'cause_desordre'] = 'certitudes_causes'
df_reduit.loc[df_reduit['cause_desordre'] == "Aujourd'hui, personne ne peut dire avec certitude les vraies raisons des désordres du climat", 'cause_desordre'] = 'pas_certitude'

df_reduit.loc[df_reduit['veracite_scientifique'] == 'Les scientifiques qui étudient les évolutions du climat évaluent correctement les risques du changement climatique', 'veracite_scientifique'] = 'evalue_correctement'
df_reduit.loc[df_reduit['veracite_scientifique'] == "Les scientifiques qui étudient les évolutions du climat exagèrent les risques du changement climatique", 'veracite_scientifique'] = 'scientifique_exagere'

# Cette variable est la cible du modèle
df_reduit.loc[df_reduit['solutions'] == 'Il faudra modifier de façon importante nos modes de vie pour limiter le changement climatique', 'solutions'] = 'changer'
df_reduit.loc[df_reduit['solutions'].str.contains("le changement climatique est inévitable") , 'solutions'] = 'inevitable'
df_reduit.loc[df_reduit['solutions'] == "C'est aux Etats de rechercher un accord au niveau mondial pour limiter le changement climatique", 'solutions'] = 'entre_etats'
df_reduit.loc[df_reduit['solutions'] == "Le progrès technique permettra de trouver des solutions pour limiter le changement climatique", 'solutions'] = 'progres'

In [ ]:
# pour faire un modèle de classification binaire simplifié
df_reduit.loc[df_reduit['solutions'] == 'entre_etats', 'solutions'] = 'ne_pas_changer'
df_reduit.loc[df_reduit['solutions'] == 'inevitable', 'solutions'] = 'ne_pas_changer'
df_reduit.loc[df_reduit['solutions'] == 'progres', 'solutions'] = 'ne_pas_changer'

df_reduit = df_reduit[df_reduit['solutions'].isin(["changer","ne_pas_changer"])]


## Modélisation

In [ ]:
# y est la variable à prédire et X les variables explicatives
y = df_reduit["solutions"]
X = df_reduit.drop(["solutions","number_of_NaNs","Vague"], axis=1)

In [ ]:
#Encodage des variables : conversion des variables catégorielles en numérique
features_cat = [e for e in features if e not in ("solutions", "Vague")]

encoder = OrdinalEncoder(
    cols=features_cat,
        return_df=True).fit(X)

X=encoder.transform(X)

In [ ]:
#Encodage de la varible cible
encoder_target = OrdinalEncoder(
    cols=["solutions"],
        return_df=True).fit(y)

y=encoder_target.transform(y)

In [ ]:
#Split train/test et modélisation
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, train_size=0.75, random_state=1
)

model = catboost.CatBoostClassifier(scale_pos_weight=1.2, max_depth=5)
model.fit(Xtrain, ytrain, verbose=False)

## Utilisation de Shapash

In [ ]:
#label pour Shapash
response_dict = {1: 'ne_pas_changer', 2: 'changer'}

In [ ]:
#Init Shapash
xpl = SmartExplainer(model=model,preprocessing=encoder, label_dict=response_dict) 

In [ ]:
# Compilation de Shapash
xpl.compile(
    x=Xtest,    
    y_target = ytest,
    additional_data=df_reduit.loc[Xtest.index, :][["Vague"]],
)

In [ ]:
xpl.plot.confusion_matrix_plot()

In [ ]:
xpl.plot.lift_curve_plot()

# Explicabilité

Dans cette section, nous allons pouvoir utiliser l'ensemble des méthodes présentées en cours

In [ ]:
#Lancement de la webapp
app=xpl.run_app(host='0.0.0.0',port='8084')


xpl.init_app()
app = xpl.smartapp.app
app.run_server(debug=False, port=8079)


In [ ]:
app.kill()

In [ ]:
xpl.plot.features_importance(mode='global-local', max_features=10, zoom=True)

# Application sur le sondage framaform

L'idée est de pouvoir tester les prédictions du modèle et son explicabilité sur vos résultats. 
Cela donne un atre axe de compréhension des résultats de l'explicabilité

In [ ]:
df_framaform = pd.read_csv("./data/Sondage_TP_barometre.csv", encoding = 'utf-8', sep='\t')
df_framaform = df_framaform[df_framaform['Vague']=='TP_DUT2026']

## transformation des données
Il faut reproduire les mêmes préparation, tranformations que sur le dataset initial

In [ ]:
# création d'une variable sur une note moyenne des mesures que le sondé considère souhaitable
notes_mapping = {
    'Très souhaitable': 4,
    'Assez souhaitable': 3,
    'Pas vraiment souhaitable': 2,
    'Pas du tout souhaitable': 1
}
def calculer_moyenne(row):
    notes = [notes_mapping[val] for val in row if val != 'No rep.']
    return sum(notes) / len(notes) if notes else 0

df_framaform_q21 = df_framaform.filter(like='q21', axis=1).replace({'nan': 'No rep.', np.nan: 'No rep.'})

colonnes_mesures_souhaitables = df_framaform_q21.filter(like='q21')

df_framaform_q21['moyenne_mesures_souhaitables'] = colonnes_mesures_souhaitables.apply(calculer_moyenne, axis=1)

df_framaform_q21['moyenne_mesures_souhaitables'].round().value_counts()

In [ ]:
df_framaform = pd.concat([df_framaform, df_framaform_q21['moyenne_mesures_souhaitables']], axis=1)

In [ ]:
df_framaform_reduit = df_framaform.copy()

df_framaform_reduit.rename(columns=columns_to_rename, inplace=True)

In [ ]:
df_framaform_reduit = df_framaform_reduit[features]

In [ ]:
df_framaform_reduit = df_framaform_reduit.fillna("No rep.", inplace=True)

In [ ]:
df_framaform_reduit.loc[df_framaform_reduit['cause_changement_climatique'] == 'Le réchauffement de la planète est causé par les activités humaines', 'cause_changement_climatique'] = 'activites_humaines'
df_framaform_reduit.loc[df_framaform_reduit['cause_changement_climatique'] == "Il s'agit uniquement d'un phénomène naturel qui a toujours existé", 'cause_changement_climatique'] = 'phenomene_naturel'

df_framaform_reduit.loc[df_framaform_reduit['impact_effet_de_serre'] == '...ou bien une certitude pour la plupart des scientifiques ?', 'impact_effet_de_serre'] = 'certitude_scientifique'
df_framaform_reduit.loc[df_framaform_reduit['impact_effet_de_serre'] == "...une hypothèse sur laquelle les scientifiques ne sont pas tous d'accord ?", 'impact_effet_de_serre'] = 'hypothese_non_certaine'

df_framaform_reduit.loc[df_framaform_reduit['sympathie_ecologie'] == '...assez de sympathie ?', 'sympathie_ecologie'] = 'assez'
df_framaform_reduit.loc[df_framaform_reduit['sympathie_ecologie'] == "...peu de sympathie ?", 'sympathie_ecologie'] = 'peu'
df_framaform_reduit.loc[df_framaform_reduit['sympathie_ecologie'] == "...beaucoup de sympathie ?", 'sympathie_ecologie'] = 'beaucoup'
df_framaform_reduit.loc[df_framaform_reduit['sympathie_ecologie'] == "...pas de sympathie ?", 'sympathie_ecologie'] = 'pas'

df_framaform_reduit.loc[df_framaform_reduit['cause_desordre'].str.contains('Les désordres du climat'), 'cause_desordre'] = 'certitudes_causes'
df_framaform_reduit.loc[df_framaform_reduit['cause_desordre'].str.contains("Aujourd'hui, personne ne peut dire avec certitude les vraies raisons des désordres du climat"), 'cause_desordre'] = 'pas_certitude'

df_framaform_reduit.loc[df_framaform_reduit['veracite_scientifique'].str.contains('évaluent correctement les risques du changement climatique'), 'veracite_scientifique'] = 'evalue_correctement'
df_framaform_reduit.loc[df_framaform_reduit['veracite_scientifique'] == "Les scientifiques qui étudient les évolutions du climat exagèrent les risques du changement climatique", 'veracite_scientifique'] = 'scientifique_exagere'

# Cette variable est la cible du modèle
df_framaform_reduit.loc[df_framaform_reduit['solutions'] == 'Il faudra modifier de façon importante nos modes de vie pour limiter le changement climatique', 'solutions'] = 'changer'
df_framaform_reduit.loc[df_framaform_reduit['solutions'].str.contains("le changement climatique est inévitable") , 'solutions'] = 'inevitable'
df_framaform_reduit.loc[df_framaform_reduit['solutions'] == "C'est aux Etats de rechercher un accord au niveau mondial pour limiter le changement climatique", 'solutions'] = 'entre_etats'
df_framaform_reduit.loc[df_framaform_reduit['solutions'] == "Le progrès technique permettra de trouver des solutions pour limiter le changement climatique", 'solutions'] = 'progres'

In [ ]:
# pour faire un modèle de classification binaire simplifié
df_framaform_reduit.loc[df_framaform_reduit['solutions'] == 'entre_etats', 'solutions'] = 'ne_pas_changer'
df_framaform_reduit.loc[df_framaform_reduit['solutions'] == 'inevitable', 'solutions'] = 'ne_pas_changer'
df_framaform_reduit.loc[df_framaform_reduit['solutions'] == 'progres', 'solutions'] = 'ne_pas_changer'


In [ ]:
df_framaform_reduit = df_framaform_reduit[df_framaform_reduit['solutions'].isin(["changer","ne_pas_changer"])]

In [ ]:
y_framaform = df_framaform_reduit["solutions"]
X_framaform = df_framaform_reduit.drop(["solutions","Vague"], axis=1)

In [ ]:
X_framaform=encoder.transform(X_framaform)
y_framaform=encoder_target.transform(y_framaform)

In [ ]:
prediction_framaform = model.predict(X_framaform)


# Recalcul local pour eviter une variable stale
prediction_framaform = model.predict(X_framaform)

# Mise a plat
y_true_num = np.array(y_framaform).ravel()
y_pred_num = np.array(prediction_framaform).ravel()

y_true_txt = encoder_target.inverse_transform(
    pd.DataFrame({"solutions": y_true_num})
)["solutions"].to_numpy()

y_pred_txt = encoder_target.inverse_transform(
    pd.DataFrame({"solutions": y_pred_num})
)["solutions"].to_numpy()

In [ ]:
response_dict = {1: 'ne_pas_changer', 2: 'changer'}

In [ ]:
xpl_framaform = SmartExplainer(model=model,preprocessing=encoder, label_dict=response_dict) 

In [ ]:
#Si pas assez de données framaform, concat avec Xtest
random_indices = Xtest.sample(n=500, random_state=42).index

X_frama = pd.concat([X_framaform, Xtest.loc[random_indices]])
y_frama = pd.concat([y_framaform, ytest.loc[random_indices]])

additional_data_frama = pd.concat([
    df_framaform[["Vague", "id_user"]],
    df_reduit.loc[random_indices, ["Vague"]]
])


#X_frama = X_framaform
#y_frama = y_framaform
#additional_data_frama=df_framaform[["Vague","id_user"]]

In [ ]:
xpl_framaform.compile(
    x=X_frama,    
    y_target = y_frama,
    additional_data=additional_data_frama,
)

In [ ]:
app_framaform=xpl_framaform.run_app(port=8080)

In [ ]:
xpl_framaform.plot.confusion_matrix_plot()

In [ ]:
xpl_framaform.plot.compare_plot(index=[df_framaform_reduit.index[0], df_framaform_reduit.index[1]])